# 30 DPO: prefer the cheaper correct query (A100)

Starts from the SFT model kept on validation by R-VES (`25_sft_evaluate`: the method, then epoch 1 or 2). Preference pairs come from the train pool, only for questions where two correct queries differ at least 2x in cost (about 2,200 questions): up to 4 pairs of a cheaper correct query over a pricier correct one, plus 1 pair of a cheapest correct query over a wrong one, so speed never wins over correctness. The loss is DPO plus the chosen query's cross entropy (keeps correct answers likely). The reference model is the starting SFT model; its log probabilities are computed once at the start.

1. **Export** the pairs (CPU work, a few minutes; skipped when it exists).
2. **Train** with the settings in the config cell (this run: learning rate 5e-5, 2 epochs, about 280 steps of 16 questions). The log shows the pair loss and the reward accuracy (share of pairs where the model now prefers the chosen query more than the SFT model did); the same on the validation pairs at the start, every 50 steps and at the end.
3. **Predict** validation, dev and mini-dev with the DPO adapter (and the same target, for a tagged model).

**Runtime:** A100, about 1 hour 15 minutes (training about an hour; the reference pass is reused from the first run). **Resumable:** after a disconnect, run the setup and config cells again, then the cell that was running (checkpoints every 25 steps; the reference log probabilities are saved as they are computed).

**After every code update, reopen this notebook from the GitHub link.**

In [ ]:
# ---- effsql setup cell: identical in every notebook (source: notebooks/setup_cell.py) ----
REPO_URL = "https://github.com/selimsidan/correct-is-not-enough"
REPO_DIR = "/content/correct-is-not-enough"
REF = "main"  # branch, tag or commit hash to run
GPU = True  # set True in GPU notebooks: installs the [gen] extra (vLLM)
DRIVE_ROOT = "/content/drive/MyDrive/586"
LOCAL_ROOT = "/content/bird_local"

import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")


def sh(*cmd, check=True):
    r = subprocess.run(cmd, text=True, capture_output=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"{' '.join(cmd)} failed:\n{r.stderr}")
    return r.stdout.strip()


if not os.path.isdir(REPO_DIR):
    sh("git", "clone", "--quiet", REPO_URL, REPO_DIR)
sh("git", "-C", REPO_DIR, "fetch", "--quiet", "--all", "--tags")
sh("git", "-C", REPO_DIR, "checkout", "--quiet", REF)
if sh("git", "-C", REPO_DIR, "rev-parse", "--verify", "--quiet", f"origin/{REF}", check=False):
    sh("git", "-C", REPO_DIR, "merge", "--ff-only", "--quiet", f"origin/{REF}")
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", REPO_DIR + ("[gen]" if GPU else "")], check=True
)
os.environ["EFFSQL_DRIVE_ROOT"] = DRIVE_ROOT
os.environ["EFFSQL_LOCAL_ROOT"] = LOCAL_ROOT
print("effsql commit:", sh("git", "-C", REPO_DIR, "rev-parse", "HEAD"))

In [ ]:
# Training libraries (transformers and peft, pinned in pyproject.toml); vLLM above brings torch.
!pip install -q -e "{REPO_DIR}[train]"
# Colab ships torchao 0.10.0, which peft rejects with an error (it needs 0.16 or newer). We do not use it.
!pip uninstall -q -y torchao
import importlib.metadata as md
print({p: md.version(p) for p in ("torch", "transformers", "peft", "vllm")})

In [ ]:
import json, os
POOL, EXPORT, DPO_EXPORT = "pool_v1", "sft_v1", "dpo_v1"
R = f"{DRIVE_ROOT}/effsql/runs"
# The SFT model kept on validation by R-VES (25_sft_evaluate: method, then epoch 1 or 2): DPO starts from it.
CHOSEN = json.load(open(f"{R}/{EXPORT}_epoch_choice_rves/selection.json"))["winner"]
cm = json.load(open(f"{R}/{CHOSEN}/adapter_meta.json"))
INIT, TAG = cm.get("adapter_run", CHOSEN), cm.get("tag")  # TAG only for a tagged model
TAG_ARG = f"--tag={TAG}" if TAG else ""
METHOD = f"dpo-{cm['arm']}" + (f"-tag{TAG}" if TAG else "")
# DPO settings. The first run (train.yaml defaults: lr 1e-5, 1 epoch) is dpo_v1_best: its pair loss barely
# moved. This run: 5x the learning rate (LoRA scale, half the SFT rate) and 2 epochs, same pairs and beta.
NAME, SETTINGS = "strong", "--lr 5e-5 --epochs 2"
FIRST = f"{DPO_EXPORT}_{cm['arm']}"
RUN = f"{FIRST}_{NAME}" if NAME else FIRST
METHOD += f"-{NAME}" if NAME else ""
print("SFT model:", CHOSEN, "| adapter:", INIT, "| target:", TAG, "| DPO run:", RUN, "|", SETTINGS)
!nvidia-smi --query-gpu=name,memory.total --format=csv

### 1. Export the preference pairs (skipped when it already exists)

In [ ]:
if not os.path.exists(f"{R}/{DPO_EXPORT}/export_meta.json"):
    !python -m effsql.train export --pool-run {POOL} --run {DPO_EXPORT} --arms dpo
else:
    print("export exists:", DPO_EXPORT)

### 2. Train

In [ ]:
# The reference log probabilities depend only on the starting model and the pairs: reuse the first run's.
import shutil
if RUN != FIRST and os.path.exists(f"{R}/{FIRST}/ref_logps.jsonl") and not os.path.exists(f"{R}/{RUN}/ref_logps.jsonl"):
    os.makedirs(f"{R}/{RUN}", exist_ok=True)
    shutil.copy(f"{R}/{FIRST}/ref_logps.jsonl", f"{R}/{RUN}/ref_logps.jsonl")
!python -m effsql.train dpo --export {DPO_EXPORT} --run {RUN} --init-from {INIT} {TAG_ARG} {SETTINGS}

### 3. Predict validation, dev and mini-dev

In [ ]:
!python -m effsql.train predict --run {RUN} --splits val,dev,minidev

### Done: disconnect now, then run `31_dpo_evaluate` on a CPU runtime